# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema and available at:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step will download the metadata and allow you to inspect its structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Dataset Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Description: {metadata.description}\n")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")


## 2. Data Overview
Review available record sets and the fields (columns) within each. All entities are referenced by their `@id` fields.

_Hint: For Croissant datasets, use the `dataset.record_sets` property to enumerate record sets, and for each record set, get the fields and their respective columns._

In [ ]:
print("Record Sets Overview:")

record_set_ids = []
for rs in dataset.record_sets:
    print(f"- Record Set: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields (columns):")
    for field in rs.fields:
        print(f"    - {getattr(field, 'name', None)} (@id: {field.id}, dataType: {getattr(field, 'data_type', None)})")
    print("")

# Optionally, show how many record sets and their @ids for reference later
print(f"Found {len(record_set_ids)} record set(s): {record_set_ids}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames. For reproducibility, always reference entities by their `@id`.

_In this dataset, there is likely a main record set containing protein data; adjust the `record_sets_to_load` list as available from the above cell._

In [ ]:
# List the record sets by @id for loading (from above output)
record_sets_to_load = record_set_ids  # Use all, or subset if desired
dataframes = {}

for rs_id in record_sets_to_load:
    print(f"Loading records for record set {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f" - DataFrame columns: {df.columns.tolist()}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply some basic data processing: filter on a numeric field, normalize it, and group by a key categorical field.

_Replace the record set and field `@id` values with those found in your dataset. The code below demonstrates the process on the first record set found._

In [ ]:
# Choose a record set and field for EDA
target_record_set = record_set_ids[0] if record_set_ids else None
df = dataframes[target_record_set]

# List numeric columns by checking their data type in the record set definition
numeric_fields = []
groupable_fields = []
for rs in dataset.record_sets:
    if rs.id == target_record_set:
        for field in rs.fields:
            if getattr(field, 'data_type', '').lower() in ['integer', 'float', 'number']:
                numeric_fields.append(field.id)
            # Try to find some groupable field (e.g., protein family, etc.)
            if getattr(field, 'data_type', '').lower() in ['text', 'string']:
                groupable_fields.append(field.id)

# Pick the first numeric field found
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Chosen numeric field for filtering/normalizing: {numeric_field}")
else:
    print("No numeric fields found for EDA.")

# Pick the first groupable text field found
group_field = groupable_fields[0] if groupable_fields else None

if numeric_fields:
    # EDA: filter for values > threshold, normalize, group
    threshold = 10

    # Be sure the field exists in DataFrame columns
    if numeric_field in df.columns:
        filtered_df = df[df[numeric_field].astype(float) > threshold].copy()
        print(f"Filtered records in '{target_record_set}' where {numeric_field} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()

        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print(f"Field {numeric_field} not found in DataFrame columns.")
else:
    print("No numeric fields to perform EDA.")


## 5. Visualization
Visualize the distribution of the chosen numeric field (e.g., histogram) and show a boxplot by group if appropriate.

_Below, basic visualizations are created using matplotlib and seaborn. You must have the fields selected in the previous sections._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize histogram of the normalized field
if 'filtered_df' in locals() and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].astype(float), bins=30, kde=True)
    plt.title(f'Histogram of {numeric_field} after filtering')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f'Boxplot of {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No numeric field or filtered data available.")

## 6. Conclusion
In this notebook, we demonstrated the use of the [`mlcroissant`](https://mlcommons.github.io/croissant/) library to:

- Load Croissant metadata from a schema-defined dataset
- Enumerate record sets, fields, and their unique `@id`s
- Extract tabular data dynamically using entity `@id`
- Apply exploratory data operations, such as filtering, normalization, and grouping
- Visualize data distributions

The [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset enables reproducible, semantic exploration and analysis of mass spectrometry results for extracellular vesicle proteins.

> **Next Steps:** Refine field choices for EDA based on biological hypotheses, explore joining with protein ontologies, or integrate into downstream ML workflows. Data provenance and field lineages are all traceable via `@id`s per ML Croissant best practice.